In [0]:
-- Count total users. 
select count(user_id) from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;


-- Count users by gender. 
select gender,count(user_id) as count
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
group by gender;

-- Average time_on_site per device_type. 
select device_type,avg(time_on_site) as avg_time
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
group by device_type;

-- Total purchases by device_type. 
select device_type,sum(purchase) as total
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
group by device_type;

-- Count users who saw discount vs not. 
select discount_seen,count(user_id) as count
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
group by discount_seen;

/*Create a column: 
"High Engagement" if pages_viewed > 5 
else "Low Engagement" 
*/
select user_id,
case when pages_viewed > 5 then "High Engagement" 
  else "Low Engagement" end as engagement
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;

/*Categorize users: 
"Buyer" if purchase = 1 
else "Non-Buyer" 
*/
select user_id,
case when purchase = 1 then "Buyer" 
  else "Non-Buyer" end as customer_type
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;

/*Create discount effectiveness: 
"Effective" if discount_seen = 1 AND purchase = 1 */
select user_id,
case when discount_seen = 1 AND purchase = 1 then "Effective" 
  else "Not Effective" end as effect
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;

-- Avg time_on_site for buyers vs non-buyers 
with t as (
  select *,
  case when purchase = 1 then 'Buyer'
  else 'Non-Buyer' end as customer_type
  from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
)
select customer_type,avg(time_on_site) as avg_time
from t
group by customer_type;

-- Bounce rate avg by device_type 
select device_type, avg(bounce_rate) as avg_bounce
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
group by device_type;

/*Calculate conversion rate: 
total purchases / total users
Conversion rate by device_type */
select device_type, sum(purchase)/count(user_id) as conversion_rate
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
group by device_type;

/*Conversion rate for returning vs new users */
with t as (
  select *,
  case when returning_user = 1 then 'return_user'
  else 'new_user' end as user_type
  from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
)
select user_type,sum(purchase)/count(user_id) as conversion_rate
from t
group by user_type;

/*Find top 5 users with highest time_on_site */
with t as (
  select user_id, sum(time_on_site) as total_time,
    rank()over(order by sum(time_on_site) desc) as rnk
  from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
  group by user_id
)
select user_id,total_time,rnk
from t
where rnk <=5;

/*Find users with high engagement but no purchase
(pages_viewed > 5 AND purchase = 0) 
*/

select user_id 
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
where pages_viewed > 5 AND purchase = 0; 

/*Find correlation-like insight: 
Avg pages_viewed for buyers vs non-buyers 
*/
with t as (
  select user_id,pages_viewed,
  case when purchase = 1 then "Buyer" 
  else "Non-Buyer" end as user_type
  from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
)
select user_type,avg(pages_viewed) as avg_pages
from t
group by user_type;

/*Find users who: 
clicked ad 
saw discount 
but didn’t purchase 
*/
select user_id
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
where ad_clicked = 1 AND discount_seen = 1 AND purchase = 0;

/*Find % of users who: 
added items to cart but didn’t purchase 
*/
select (
  select count(user_id) 
  from parquet.`/Volumes/data/orders/files/result/eco.parquet/` 
  where cart_items > 0 AND purchase = 0 )
    /count(user_id) as per
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;

-- Running average of pages_viewed by user_id 
select 
  user_id,
  avg(pages_viewed) over (partition by user_id order by avg_session_time
    rows between unbounded preceding and current row
  ) as running_avg_pages
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;

--Find top user per device_type 
with t as (
  select user_id,device_type,time_on_site,
  rank()over(partition by device_type order by time_on_site desc) as rnk
  from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
)
select user_id,device_type,time_on_site
from t 
where rnk = 1;

--Rank users by time_on_site 
select user_id,time_on_site,rank()over(order by time_on_site desc) as rnk
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;

-- Dense rank users by avg_session_time 
select user_id, avg_session_time,
  dense_rank()over(order by avg_session_time desc) as dense_rnk
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;

/*Create a summary table: 
device_type 
total_users 
conversion_rate 
avg_time_on_site */
select device_type,count(user_id) as total_users,
  sum(purchase)/count(user_id) as conversion_rate,
  avg(time_on_site) as avg_time
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`
group by device_type;

/*Daily/Session KPI simulation: 
avg_session_time 
bounce_rate 
conversion_rate */
select 
  avg(avg_session_time) as avg_session_time,
  avg(bounce_rate) as avg_bounce_rate,
  sum(purchase)/count(user_id) as conversion_rate
from parquet.`/Volumes/data/orders/files/result/eco.parquet/`;